# Lab 00 · Inspect Markdown and build chunks

Module A • **Core, offline.** The document is real source text and the printed answers are deterministic.

**Workflow:** read → predict → run → edit the marked variable → rerun → answer the linked questions in `assessments/quiz.html`.  \n**No answer needs a screenshot.** Keep your notebook output visible while taking the quiz.

# Your task board · Run first, answer later

Use the teaching cells below to determine the answers. For an **EDIT** task, change the marked variable and rerun the specified cell. The hints and full answers are intentionally **open at the end of this notebook**. Try the questions first.

**A1 · NAT · after cell 00 · A1** — How many second-level Markdown headings (##) are in field_notes.md?  
NAT: `_________`

**A2 · NAT · after cell 00 · A1** — How many paragraphs are found by the blank-line splitter?  
NAT: `_________`

**A3 · NAT · after cell 00 · A1** — How many whitespace-separated words are in the entire Markdown file?  
NAT: `_________`

**A4 · NAT · after cell 00 · A2** — With CHUNK_WORDS=40 and OVERLAP_WORDS=10, how many chunks?  
NAT: `_________`

**A5 · NAT · after cell 00 · A2** — At the default setting, how many words are in the final chunk?  
NAT: `_________`

**A6 · NAT · after cell 00 · A4 EDIT** — After changing CHUNK_WORDS to 60, with overlap 10, how many chunks?  
NAT: `_________`

**A7 · NAT · after cell 00 · A4 EDIT** — With CHUNK_WORDS=40 and OVERLAP_WORDS=0, how many chunks?  
NAT: `_________`

**A8 · MCQ · after cell 00 · A2** — Which statement about the word-window chunker is true?
  - **A.** It counts Gemini model tokens
  - **B.** It uses whitespace-separated words
  - **C.** It creates model embeddings
  - **D.** It guarantees sentences remain intact

---

**Assessment:** [Interactive self-check](../assessments/quiz.html?module=A) · [Full question sheet](../assessments/QUESTION_SHEET.md). You can learn without grading; 70% unlocks the next quiz on the portal.

**Official documentation:** [LangChain text splitters](https://docs.langchain.com/oss/python/integrations/splitters/index) · [Gemini embeddings](https://ai.google.dev/gemini-api/docs/embeddings).

In [ ]:
from pathlib import Path
import numpy as np
from IPython.display import display

def find_workshop_root():
    candidates = [Path.cwd(), *Path.cwd().parents, Path('/content/embedding_workshop')]
    for p in candidates:
        if (p / 'data' / 'field_notes.md').is_file():
            return p
    raise FileNotFoundError('Cannot find data/field_notes.md. Unzip the complete workshop, not just one notebook. Check README.md.')
ROOT = find_workshop_root()
print('Workshop root:', ROOT)


## A1 · Inspect before chunking
**Predict:** How many sections begin with a `##` heading? How many whitespace-separated words does the whole file contain? Then run the next cell.

In [ ]:
from collections import Counter
import re
raw = (ROOT / 'data/field_notes.md').read_text(encoding='utf-8')
section_titles = re.findall(r'^## (.+)$', raw, flags=re.MULTILINE)
paragraphs = [p.strip() for p in re.split(r'\n\s*\n', raw) if p.strip()]
print('heading_count =', len(section_titles))
print('headings =', section_titles)
print('paragraph_count =', len(paragraphs))
print('whitespace_word_count =', len(raw.split()))
print('characters =', len(raw))

## A2 · Implement a reproducible word-window chunker
A *word* here means a whitespace-separated token for teaching purposes, **not** model-tokenization. The function keeps source order and measures overlap in words. The windows intentionally strip Markdown formatting.

In [ ]:
def word_chunks(text, chunk_words, overlap_words):
    if not 0 <= overlap_words < chunk_words:
        raise ValueError('Need 0 <= overlap_words < chunk_words')
    words = text.split()
    stride = chunk_words - overlap_words
    return [' '.join(words[start:start + chunk_words])
            for start in range(0, len(words), stride)]

# ✏️ EDIT ONLY THESE TWO LINES for each experiment.
CHUNK_WORDS = 40
OVERLAP_WORDS = 10
chunks = word_chunks(raw, CHUNK_WORDS, OVERLAP_WORDS)
print('chunk_count =', len(chunks))
print('chunk_lengths =', [len(x.split()) for x in chunks])
print('first_chunk =', chunks[0])
print('last_chunk =', chunks[-1])

## A3 · Verify the overlap instead of trusting the number
The last `OVERLAP_WORDS` in chunk 0 should equal the first `OVERLAP_WORDS` in chunk 1. Run this cell.

In [ ]:
overlap_tail = chunks[0].split()[-OVERLAP_WORDS:]
overlap_head = chunks[1].split()[:OVERLAP_WORDS]
print('actual_overlap_words =', len(overlap_tail))
print('overlap_match =', overlap_tail == overlap_head)
print('overlap_text =', ' '.join(overlap_tail))
assert overlap_tail == overlap_head

## A4 · Perform two controlled edits
1. Change `CHUNK_WORDS` to **60** and keep overlap at **10**. Run A2 again. Answer A6 in the quiz.
2. Restore size **40**, change `OVERLAP_WORDS` to **0**. Run A2 again. Answer A7.
3. Restore **40 / 10** before moving on.

**Think:** overlap changes storage/count; it does not magically supply semantics.

In [ ]:
# Independent check: edit parameters in A2, not here.
print('current configuration:', CHUNK_WORDS, OVERLAP_WORDS)
print('expected stride:', CHUNK_WORDS - OVERLAP_WORDS)
print('last chunk word count:', len(chunks[-1].split()))

### Finish Module A
Open `assessments/quiz.html` → **A · Chunks**. The quiz is marked automatically. You need **70%** to unlock the next module in the portal.

## Hints · Use these only after attempting the tasks

- **A1:** Try `re.findall(r"^## ", raw, flags=re.MULTILINE)`.
- **A2:** Inspect the list created by the blank-line `re.split`.
- **A3:** `len(raw.split())` counts whitespace-separated words, not tokens.
- **A4:** Stride is window size minus overlap. Count start offsets.
- **A5:** Look at the last entry of `chunk_lengths`.
- **A6:** Use the edited stride, not the default stride.
- **A7:** Remove overlap; now each start advances one whole window.
- **A8:** Find the exact operation used to create `words`.

These are learning hints, not the solutions. Scroll down for the fully open answer key.


## Open answers · Full worked key and explanations

The answers below refer to the **supplied dataset and stated default/EDIT settings**. If you changed additional inputs, your results may differ. Compare the experiment and formula—not just the number.

**A1: 5**  
Regex ^## matches five subsection headings.

**A2: 17**  
The teaching file contains 17 nonempty blank-line-separated blocks.

**A3: 281**  
The deterministic metric is len(raw.split()), not model tokens.

**A4: 10**  
Stride = 40-10 = 30; 281 words produce 10 start positions.

**A5: 11**  
Last window starts at word offset 270, leaving 11 words.

**A6: 6**  
Stride 50. Starts: 0,50,100,150,200,250.

**A7: 8**  
Stride 40. The last chunk contains only one word.

**A8: B — It uses whitespace-separated words**  
The simple example is intentionally word-window based, not a tokenizer or semantic splitter.

### Extension challenge — Interview explanation

**Challenge:** A search task fails because the answer was split at a chunk boundary. How would you prove the failure comes from chunking and not embeddings?

**Self-check:** Explain the trade-off aloud in 45 seconds using a concrete example.
